In [68]:
import pandas as pd
import numpy as np


df = pd.read_csv("./data/log_sensor.csv")
print(f"Data: {df.shape[0]} baris, {df.shape[1]} kolom")
df.head()

Data: 235 baris, 9 kolom


,Timestamp,soil_moisture,soil_temperature,air_temperature,air_humidity,nitrogen,fosfor,kalium,ec
0,2026-07-01T17:15:27Z,33.5,23.7,24.4,71.0,143,127,715,255
1,2026-07-01T17:20:27Z,31.9,23.3,24.0,71.2,138,122,689,246
2,2026-07-01T17:25:27Z,31.2,23.1,23.6,73.2,136,120,676,242
3,2026-07-01T17:30:27Z,30.6,22.9,23.5,74.4,134,119,663,239
4,2026-07-01T17:35:27Z,30.0,22.7,23.5,73.9,134,118,657,238


# Cek data

In [69]:
df[["soil_moisture", "soil_temperature", "air_temperature",
    "air_humidity", "nitrogen", "fosfor", "kalium", "ec"]].describe().round(2)

,soil_moisture,soil_temperature,air_temperature,air_humidity,nitrogen,fosfor,kalium,ec
count,235.00,235.00,235.00,235.00,235.00,235.00,235.00,235.00
mean,29.30,22.66,25.20,66.79,125.79,111.16,591.89,223.83
std,1.32,2.36,2.71,8.48,14.03,12.66,112.02,25.38
min,27.30,19.60,21.50,50.60,96.00,84.00,338.00,170.00
25%,29.20,20.50,22.40,58.65,104.00,91.50,421.00,184.50
50%,29.50,22.30,25.80,69.30,131.00,116.00,640.00,234.00
75%,29.70,24.80,27.75,74.50,135.00,120.00,665.00,241.00
max,36.80,28.10,29.70,77.40,143.00,127.00,715.00,255.00


# Normalisasi Skala Sensor

In [70]:
# ec: sensor output µS/cm, konversi ke mS/cm (bagi 100)

df["ec"] = df["ec"] / 100.0       

# df["kalium"] = df["kalium"] / 3.5

print("Setelah normalisasi:")
print(df[["nitrogen", "fosfor", "kalium", "ec"]].describe().round(2))

Setelah normalisasi:
       nitrogen  fosfor  kalium      ec
count    235.00  235.00  235.00  235.00
mean     125.79  111.16  591.89    2.24
std       14.03   12.66  112.02    0.25
min       96.00   84.00  338.00    1.70
25%      104.00   91.50  421.00    1.85
50%      131.00  116.00  640.00    2.34
75%      135.00  120.00  665.00    2.41
max      143.00  127.00  715.00    2.55


# Tambah Kolom Fase (butuh plant_age)

In [71]:
TANGGAL_TANAM = pd.Timestamp("2026-05-01", tz="UTC")

df["Timestamp"] = pd.to_datetime(df["Timestamp"])
df["plant_age"] = (df["Timestamp"] - TANGGAL_TANAM).dt.days

def get_fase(age):
    if age <= 30:   return 0    # establishment
    elif age <= 55: return 1    # vegetatif
    elif age <= 75: return 2    # berbunga (initial flowering - fruit set)
    else:           return 3    # pematangan (fruit development & maturation)

df["fase"] = df["plant_age"].apply(get_fase)

FASE_NAMES = {0: "establishment", 1: "vegetatif", 2: "berbunga", 3: "pematangan"}

print(df[["Timestamp", "plant_age", "fase"]].head())
print("\nDistribusi fase:")
for code, name in FASE_NAMES.items():
    n = (df["fase"] == code).sum()
    print(f"  {code} = {name:15s} {n}")

                  Timestamp  plant_age  fase
0 2026-07-01 17:15:27+00:00         61     2
1 2026-07-01 17:20:27+00:00         61     2
2 2026-07-01 17:25:27+00:00         61     2
3 2026-07-01 17:30:27+00:00         61     2
4 2026-07-01 17:35:27+00:00         61     2

Distribusi fase:
  0 = establishment   0
  1 = vegetatif       0
  2 = berbunga        235
  3 = pematangan      0


# Rasio Ideal NPK per Fase (dari Haifa)

In [72]:
# Rasio N:P2O5:K2O ideal per fase (Haifa Crop Guide)
RASIO_IDEAL = {
    0: (1, 2, 1),   # establishment - P dominan (root development)
    1: (1, 1, 1),   # vegetatif - seimbang
    2: (2, 1, 3),   # berbunga - K dominan, P turun
    3: (2, 1, 3),   # pematangan - K dominan
}

# Batas absolut per unsur (mg/kg) - sesuaikan skala sensormu
BATAS_MIN = {"n": 40, "p": 50, "k": 60}
BATAS_MAX = {"n": 150, "p": 150, "k": 200}

print("Rasio ideal & batas siap.")

Rasio ideal & batas siap.


# Labelling dataset pupuk

In [73]:
LABEL_NAMES = {
    0: "Tidak perlu",
    1: "Urea/ZA",              # N kurang
    2: "SP-36",                # P kurang
    3: "KCl",                  # K kurang
    4: "Urea/ZA + SP-36",      # N,P kurang
    5: "Urea/ZA + KCl",        # N,K kurang
    6: "SP-36 + KCl",          # P,K kurang
    7: "Urea/ZA + SP-36 + KCl",# semua kurang
    8: "NPK 15-15-15",         # maintenance
    9: "Kurangi pemupukan N",  # N berlebih
    10: "Flush air (EC/nutrisi tinggi)",  # over/salinitas
}

def label_pupuk(row):
    n, p, k = row["nitrogen"], row["fosfor"], row["kalium"]
    ec = row["ec"]
    fase = row["fase"]   # sekarang int (0-3)

    # --- Cek kelebihan dulu ---
    if ec > 4.0:
        return 10
    if n > BATAS_MAX["n"] and k > BATAS_MAX["k"]:
        return 10
    if n > BATAS_MAX["n"]:
        return 9

    # --- Defisiensi berdasarkan rasio fase ---
    rn, rp, rk = RASIO_IDEAL[fase]   # akses pakai int
    total_ratio = rn + rp + rk

    total_npk = n + p + k
    if total_npk == 0:
        return 7

    prop_n, prop_p, prop_k = n/total_npk, p/total_npk, k/total_npk
    ideal_n, ideal_p, ideal_k = rn/total_ratio, rp/total_ratio, rk/total_ratio

    TOLERANSI = 0.7
    n_low = (prop_n < ideal_n * TOLERANSI) or (n < BATAS_MIN["n"])
    p_low = (prop_p < ideal_p * TOLERANSI) or (p < BATAS_MIN["p"])
    k_low = (prop_k < ideal_k * TOLERANSI) or (k < BATAS_MIN["k"])

    if   n_low and p_low and k_low: return 7
    elif n_low and p_low:           return 4
    elif n_low and k_low:           return 5
    elif p_low and k_low:           return 6
    elif n_low:                     return 1
    elif p_low:                     return 2
    elif k_low:                     return 3
    else:                           return 0

print("Fungsi label_pupuk() siap.")

Fungsi label_pupuk() siap.


In [74]:
df["recommendation"] = df.apply(label_pupuk, axis=1)
df["recommendation_label"] = df["recommendation"].map(LABEL_NAMES)

print("Distribusi rekomendasi:")
dist = df["recommendation"].value_counts().sort_index()
for code, count in dist.items():
    print(f"  {code} = {LABEL_NAMES[code]:35s} {count:5d} ({count/len(df)*100:.1f}%)")

Distribusi rekomendasi:
  1 = Urea/ZA                               235 (100.0%)


# Preview Hasil

In [75]:
df["fase_label"] = df["fase"].map(FASE_NAMES)
df[["Timestamp", "nitrogen", "fosfor", "kalium", "ec",
    "fase_label", "recommendation_label"]].head(20)

,Timestamp,nitrogen,fosfor,kalium,ec,fase_label,recommendation_label
0,2026-07-01 17:15:27+00:00,143,127,715,2.55,berbunga,Urea/ZA
1,2026-07-01 17:20:27+00:00,138,122,689,2.46,berbunga,Urea/ZA
2,2026-07-01 17:25:27+00:00,136,120,676,2.42,berbunga,Urea/ZA
3,2026-07-01 17:30:27+00:00,134,119,663,2.39,berbunga,Urea/ZA
4,2026-07-01 17:35:27+00:00,134,118,657,2.38,berbunga,Urea/ZA
5,2026-07-01 17:40:27+00:00,133,118,656,2.37,berbunga,Urea/ZA
6,2026-07-01 17:45:27+00:00,133,118,652,2.37,berbunga,Urea/ZA
7,2026-07-01 17:50:27+00:00,133,117,649,2.36,berbunga,Urea/ZA
8,2026-07-01 17:55:27+00:00,133,117,650,2.36,berbunga,Urea/ZA
9,2026-07-01 18:00:28+00:00,133,117,649,2.36,berbunga,Urea/ZA


# Simpan dataset

In [76]:
sensor_cols = ["soil_moisture", "soil_temperature", "air_temperature",
               "air_humidity", "nitrogen", "fosfor", "kalium", "ec"]

# Dataset pupuk: 7 fitur + label
cols_pupuk = ["nitrogen", "fosfor", "kalium", "plant_age", "fase", "ec", "soil_moisture", "recommendation"]
df[cols_pupuk].to_csv("data/dataset_pupuk.csv", index=False)

print("Tersimpan: dataset_pupuk.csv")
print(f"Total: {len(df)} baris")

Tersimpan: dataset_pupuk.csv
Total: 235 baris
